In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
%pip install -U langchain-text-splitters

In [ ]:
import os

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from pathlib import Path

In [16]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf)} PDF files to process")
    
    for pdf in pdf:
        print(f"\nProcessing: {pdf.name}")
        try:
            loader = PyPDFLoader(str(pdf))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
    # Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: attention.pdf


Multiple definitions in dictionary at byte 0x2251 for key /ToUnicode
Multiple definitions in dictionary at byte 0x22f8 for key /ToUnicode
Multiple definitions in dictionary at byte 0x239f for key /ToUnicode
Multiple definitions in dictionary at byte 0x2448 for key /ToUnicode
Multiple definitions in dictionary at byte 0x24f1 for key /ToUnicode
Multiple definitions in dictionary at byte 0x2598 for key /ToUnicode
Multiple definitions in dictionary at byte 0x263f for key /ToUnicode
Multiple definitions in dictionary at byte 0x26e5 for key /ToUnicode
Multiple definitions in dictionary at byte 0x278c for key /ToUnicode
Multiple definitions in dictionary at byte 0x2831 for key /ToUnicode
Multiple definitions in dictionary at byte 0x28da for key /ToUnicode
Multiple definitions in dictionary at byte 0x2981 for key /ToUnicode


  ✓ Loaded 11 pages

Processing: embeddings.pdf
  ✓ Loaded 27 pages

Processing: objectdetection.pdf
  ✓ Loaded 12 pages

Total documents loaded: 50


In [ ]:
all_pdf_documents

In [ ]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs



In [21]:
# Call the function
chunks = split_documents(all_pdf_documents)

# Display the chunks
chunks

Split 50 documents into 199 chunks

Example chunk:
Content: Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz...
Metadata: {'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiori

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

In [22]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\acer\Desktop\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager=EmbeddingManager()
embedding_manager



Loading embedding model: all-MiniLM-L6-v2


c:\Users\acer\Desktop\rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\acer\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\acer\Desktop\rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning

Model loaded successfully. Embedding dimension: 384


C:\Users\acer\AppData\Local\Temp\ipykernel_5964\3357757261.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [25]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [26]:
chunks

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

In [28]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 199 texts...


Batches: 100%|██████████| 7/7 [00:10<00:00,  1.52s/it]


Generated embeddings with shape: (199, 384)
Adding 199 documents to vector store...
Successfully added 199 documents to vector store
Total documents in collection: 199


In [30]:
rag_retriever

In [32]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.92it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [34]:




class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: Optional[float] = None
    ) -> List[Dict[str, Any]]:

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}")

        try:
            # Check collection size
            total_docs = self.vector_store.collection.count()

            print(f"Total documents in database: {total_docs}")

            if total_docs == 0:
                print("Vector database is empty!")
                return []

            # Detect distance metric
            collection_metadata = self.vector_store.collection.metadata or {}
            metric = collection_metadata.get("hnsw:space", "l2")

            print(f"Distance metric: {metric}")

            # Generate query embedding
            query_embedding = self.embedding_manager.generate_embeddings(
                [query]
            )[0]

            print(f"Query embedding shape: {query_embedding.shape}")

            # Retrieve documents from ChromaDB
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=min(top_k, total_docs),
                include=["documents", "metadatas", "distances"]
            )

            retrieved_docs = []

            documents = results["documents"][0] or []
            metadatas = results["metadatas"][0] or []
            distances = results["distances"][0] or []
            ids = results["ids"][0] or []

            for i, (doc_id, document, metadata, distance) in enumerate(
                zip(ids, documents, metadatas, distances)
            ):

                # Only calculate cosine similarity for cosine distance
                similarity_score = (
                    1 - distance if metric == "cosine" else None
                )

                # Apply threshold only when cosine similarity is available
                if (
                    score_threshold is not None
                    and similarity_score is not None
                    and similarity_score < score_threshold
                ):
                    continue

                retrieved_docs.append({
                    "id": doc_id,
                    "content": document,
                    "metadata": metadata,
                    "similarity_score": similarity_score,
                    "distance": distance,
                    "rank": i + 1
                })

            print(
                f"Retrieved {len(retrieved_docs)} documents "
                "(after filtering)"
            )

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


In [35]:
rag_retriever

In [36]:
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [37]:
results = rag_retriever.retrieve(
    query="What is attention is all you need?",
    top_k=5
)

for result in results:
    print("\nRank:", result["rank"])
    print("Distance:", result["distance"])
    print("Content:", result["content"][:300])

Retrieving documents for query: 'What is attention is all you need?'
Top K: 5
Total documents in database: 199
Distance metric: l2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.71it/s]

Generated embeddings with shape: (1, 384)
Query embedding shape: (384,)
Retrieved 5 documents (after filtering)

Rank: 1
Distance: 1.107047200202942
Content: convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,
the approach we take in our model.
As side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions
from our models and present and discuss examples in the a

Rank: 2
Distance: 1.124868392944336
Content: during training.
4 Why Self-Attention
In this section we compare various aspects of self-attention layers to the recurrent and convolu-
tional layers commonly used for mapping one variable-length sequence of symbol representations
(x1,...,x n) to another sequence of equal length (z1,...,z n), with x

Rank: 3
Distance: 1.1612920761108398
Content: described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single seque

In [38]:
results = rag_retriever.retrieve(
    query="Simple count-based embeddings",
    top_k=5
)

for result in results:
    print("\nRank:", result["rank"])
    print("Distance:", result["distance"])
    print("Content:", result["content"][:300])

Retrieving documents for query: 'Simple count-based embeddings'
Top K: 5
Total documents in database: 199
Distance metric: l2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.88it/s]

Generated embeddings with shape: (1, 384)
Query embedding shape: (384,)
Retrieved 5 documents (after filtering)

Rank: 1
Distance: 0.8596616387367249
Content: A fasttext open-source library, including pretrained embeddings for 157 languages,
is available athttps://fasttext.cc.
Another very widely used static embedding model is GloVe (Pennington et al.,
2014), short for Global Vectors, because the model is based on capturing global
corpus statistics. GloVe

Rank: 2
Distance: 0.9580503106117249
Content: context of others). We’ll then introduce theword2vecmodel family for constructing
short,densevectors that have even more useful semantic properties.
We’ll also introduce thecosine, the standard way to use embeddings to com-
putesemantic similarity, between two words, two sentences, or two documents,

Rank: 3
Distance: 0.9670743346214294
Content: 5.3 • SIMPLE COUNT-BASED EMBEDDINGS7
is traditionally followed bycherrypie, a traditional dessert
often mixed, such asstrawberryrhubarb pie. Appl

In [39]:
results = rag_retriever.retrieve(
    query="Discriminative Models",
    top_k=5
)

for result in results:
    print("\nRank:", result["rank"])
    print("Distance:", result["distance"])
    print("Content:", result["content"][:300])

Retrieving documents for query: 'Discriminative Models'
Top K: 5
Total documents in database: 199
Distance metric: l2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.53it/s]

Generated embeddings with shape: (1, 384)
Query embedding shape: (384,)
Retrieved 5 documents (after filtering)

Rank: 1
Distance: 0.9408198595046997
Content: At detection time the classiﬁers in the cascade are applied sequentially. Once
a window is classiﬁed as background the testing terminates with the background
label. If the object label is chosen, the next classiﬁer in the cascade is applied.
Only windows that are classiﬁed as object by all classiﬁer

Rank: 2
Distance: 0.9522314071655273
Content: under the object hypothesis at the optimal pose, and the conditional probability
of the window under the background hypothesis. This ratio is then compared to
a threshold τ to decide if the window contains an instance of the object
p(Xs+W |object,θ ;ηa)p(θ;ηp)
p(Xs+W |background) >τ.
When no backgro

Rank: 3
Distance: 1.0401877164840698
Content: Journal of Computer Vision 57(2) 137–154
[9] Fleuret, F., Geman, D. (2001). Coarse-to-ﬁne face detection. International
Journal of Computer Visio

### Integration Vectordb context Pipeline with LLM output

In [51]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

gsk_Yc2B6aBqWHiQdWGdyb3FYOltuMff0HzcGvv5GzjNvLjZR


In [54]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [55]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [56]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [57]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5
Total documents in database: 199
Distance metric: l2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.90it/s]


Generated embeddings with shape: (1, 384)
Query embedding shape: (384,)
Retrieved 5 documents (after filtering)


[{'id': 'doc_863012fe_39',
  'content': '2017.\n[16] Yoon Kim, Carl Denton, Luong Hoang, and Alexander M. Rush. Structured attention networks.\nIn International Conference on Learning Representations , 2017.\n[17] Diederik Kingma and Jimmy Ba. Adam: A method for stochastic optimization. In ICLR, 2015.\n[18] Oleksii Kuchaiev and Boris Ginsburg. Factorization tricks for LSTM networks. arXiv preprint\narXiv:1703.10722, 2017.\n[19] Zhouhan Lin, Minwei Feng, Cicero Nogueira dos Santos, Mo Yu, Bing Xiang, Bowen\nZhou, and Yoshua Bengio. A structured self-attentive sentence embedding. arXiv preprint\narXiv:1703.03130, 2017.\n[20] Samy Bengio Łukasz Kaiser. Can active memory replace attention? In Advances in Neural\nInformation Processing Systems, (NIPS), 2016.\n10',
  'metadata': {'producer': 'PyPDF2',
   'total_pages': 11,
   'creationdate': '',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'doc_index': 39,
   'creator': 'PyPDF',
   'description': 'Paper accepted 

In [65]:



import os
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq

# Locate and load .env
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

# Get API key
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found!")

# Initialize Groq
llm = ChatGroq(
    api_key=groq_api_key,
    model="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)

print("Groq LLM initialized successfully!")


Groq LLM initialized successfully!


In [66]:
response = llm.invoke(
    "What is self-attention? Explain in two sentences."
)

print(response.content)

Self‑attention is a mechanism that lets each element of a sequence (e.g., a word in a sentence) weigh and combine information from all other elements in the same sequence to produce a context‑aware representation of itself. It does this by computing similarity scores between the element’s query vector and the key vectors of all elements, turning those scores into attention weights that are applied to the corresponding value vectors.


In [67]:
answer = rag_simple(
    "What is attention mechanism?",
    rag_retriever,
    llm,
    top_k=3
)

print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3
Total documents in database: 199
Distance metric: l2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.38it/s]


Generated embeddings with shape: (1, 384)
Query embedding shape: (384,)
Retrieved 3 documents (after filtering)
The attention mechanism is a way for a model to dynamically weight and combine information from different positions (or elements) in a sequence. It does this by projecting each element into **queries**, **keys**, and **values**, computing similarity scores (e.g., dot‑products) between queries and keys, converting those scores into attention weights (typically with a softmax), and then taking a weighted sum of the values. This allows each position to “attend” to the most relevant parts of the input, capturing long‑range dependencies without sequential processing. In self‑attention (intra‑attention), the queries, keys, and values all come from the same sequence, enabling the model to relate different tokens within that sequence. Multi‑head attention extends this by performing several such attention operations in parallel on different sub‑spaces, enriching the representation.


In [68]:
answer = rag_simple(
    "What are the advantages of the Transformer architecture?",
    rag_retriever,
    llm,
    top_k=3
)

print(answer)

Retrieving documents for query: 'What are the advantages of the Transformer architecture?'
Top K: 3
Total documents in database: 199
Distance metric: l2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.55it/s]

Generated embeddings with shape: (1, 384)
Query embedding shape: (384,)
Retrieved 3 documents (after filtering)


The Transformer’s design gives several key benefits:

* **Highly parallelizable** – All attention and feed‑forward operations are computed simultaneously across positions, unlike recurrent models that must process tokens sequentially.  
* **Efficient multi‑head attention** – Splitting the model into h = 8 heads with reduced dimensionality (dk = dv = 64) keeps the computational cost comparable to a single‑head full‑dimensional attention while allowing the model to capture many types of relationships in parallel.  
* **Strong performance with lower cost** – On WMT 2014 EN‑DE and EN‑FR tasks it achieves higher BLEU scores than previous state‑of‑the‑art systems while requiring far fewer FLOPs (e.g., 3.3 × 10¹⁸ vs. >10¹⁹ for many baselines).  
* **Stable, deep training** – Residual connections around each sub‑layer combined with layer‑normalization make it easy to train deep stacks (N = 6 or more) without degradation.  
* **Flexible encoder‑decoder interaction** – The decoder’s encoder‑deco

In [70]:
answer = rag_simple(
    "What is object detection in details?",
    rag_retriever,
    llm,
    top_k=3
)

print(answer)

Retrieving documents for query: 'What is object detection in details?'
Top K: 3
Total documents in database: 199
Distance metric: l2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.92it/s]


Generated embeddings with shape: (1, 384)
Query embedding shape: (384,)
Retrieved 3 documents (after filtering)
Object detection is the computer‑vision task of locating and identifying all instances of one or more predefined object classes (e.g., people, cars, faces) within an image.  
It goes beyond simple image classification by not only answering “what is in the image?” but also “where is each object?” and often “how is it oriented?”.  

**Key elements of object detection**

| Aspect | Description |
|--------|-------------|
| **Goal** | Find every occurrence of the target classes, despite large variations in position, scale, and appearance. |
| **Output (pose information)** | For each detected object the system reports a pose, which may be: <ul><li>a point location</li><li>location + scale</li><li>a bounding box</li><li>a segmentation mask</li><li>more detailed part locations (e.g., eyes, nose, mouth for faces) or a 3‑D transformation relative to the camera.</li></ul> |
| **Model bu